****Análise Exploratória****

**Importando as bibliotecas**

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import plotly.graph_objects as go
import re
from IPython.display import display

**Configurando padrão visual dos gráficos**

In [18]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

**Carregando os dados processados**

In [19]:
df_matches = pd.read_parquet('../data/processed/matches_processed.parquet')
df_players = pd.read_parquet('../data/processed/players_processed.parquet')
df_editions = pd.read_parquet('../data/processed/editions_processed.parquet')

**Criando um Dataframe voltado para a análise do desempenho do Brasil**

***1) Obtendo Partidas que o Brasil jogou***

In [20]:
filtro_brasil = (df_matches['Home Team Name'] == 'Brazil') | (df_matches['Away Team Name'] == 'Brazil')
df_brasil_matches = df_matches[filtro_brasil].copy()

***2) Trazendo os jogadores***

In [21]:
df_brasil_completo = pd.merge(df_brasil_matches, df_players, on='MatchID', how='left')

***3) Trazendo o contexto da competição***

In [22]:
df_brasil_final = pd.merge(df_brasil_completo, df_editions[['Year', 'Winner', 'Avg Attendance']], on='Year', how='left')

**Bloco 1: Raio-X Ofensivo e Defensivo**

***Criando um DF isolando as partidas***

In [23]:
df_jogos_brasil = df_brasil_final.drop_duplicates(subset=['MatchID']).copy()
df_jogos_brasil['Year'] = df_jogos_brasil['Year'].astype(int)

***Obtendo gols marcados e sofrifos***

In [24]:
df_jogos_brasil['Gols Feitos'] = df_jogos_brasil.apply(
    lambda x: x['Home Team Goals'] if 'BRA' in str(x['Home Team Name']).upper() else x['Away Team Goals'], axis=1
)

df_jogos_brasil['Gols Sofridos'] = df_jogos_brasil.apply(
    lambda x: x['Away Team Goals'] if 'BRA' in str(x['Home Team Name']).upper() else x['Home Team Goals'], axis=1
)

***Agrupando resultados por copa***

In [25]:
desempenho_por_copa = df_jogos_brasil.groupby('Year')[['Gols Feitos', 'Gols Sofridos']].sum().reset_index()
desempenho_por_copa['Saldo de Gols'] = desempenho_por_copa['Gols Feitos'] - desempenho_por_copa['Gols Sofridos']

***Criando a visualização do Bloco 1***

In [26]:
fig = go.Figure()

# Cores vibrantes do Dark Mode original
cor_feitos = '#009B3A'   # Verde
cor_sofridos = '#002776' # Azul escuro
cor_saldo_pos = '#FEDF00'# Amarelo Ouro
cor_saldo_neg = '#FF0000'# Vermelho Alerta

# 1. Barra: Gols Feitos
fig.add_trace(go.Bar(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Gols Feitos'],
    name='Gols Feitos',
    marker_color=cor_feitos,
    text=desempenho_por_copa['Gols Feitos'],
    textposition='auto',
    hovertemplate='<b>%{x}</b>: %{y} Gols Feitos<extra></extra>'
))

# 2. Barra: Gols Sofridos
fig.add_trace(go.Bar(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Gols Sofridos'],
    name='Gols Sofridos',
    marker_color=cor_sofridos,
    text=desempenho_por_copa['Gols Sofridos'],
    textposition='auto',
    hovertemplate='<b>%{x}</b>: %{y} Gols Sofridos<extra></extra>'
))

# 3. Linha e Marcadores dinâmicos para o Saldo de Gols
cores_saldo = [cor_saldo_pos if saldo >= 0 else cor_saldo_neg for saldo in desempenho_por_copa['Saldo de Gols']]

fig.add_trace(go.Scatter(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Saldo de Gols'],
    name='Saldo de Gols',
    mode='lines+markers+text',
    line=dict(color='white', width=2, dash='dot'),
    marker=dict(color=cores_saldo, size=12, line=dict(color='#222222', width=1)),
    text=desempenho_por_copa['Saldo de Gols'],
    textposition='top center',
    textfont=dict(color=cores_saldo, size=14, family="Arial Black"),
    hovertemplate='Saldo: %{y}<extra></extra>'
))

# ==========================================
# NOVA REPRESENTAÇÃO DA GUERRA (DISCRETA E ELEGANTE)
# ==========================================

# 1. Adicionando uma linha de seta horizontal (Bracket) no espaço vazio (na altura Y=20 por exemplo)
fig.add_shape(
    type="line",
    x0=1939, y0=20, # Início logo após a copa de 38
    x1=1949, y1=20, # Fim logo antes da copa de 50
    line=dict(color="#666666", width=2, dash="solid"),
)
# Adicionando pequenas "perninhas" verticais para fazer o formato de colchete ] [
fig.add_shape(type="line", x0=1939, y0=19, x1=1939, y1=21, line=dict(color="#666666", width=2))
fig.add_shape(type="line", x0=1949, y0=19, x1=1949, y1=21, line=dict(color="#666666", width=2))

# 2. Adicionando o texto explicativo flutuante acima da linha
fig.add_annotation(
    x=1944, # Centro do período
    y=23,   # Um pouco acima da linha
    text="Intervalo:<br>2ª Guerra Mundial",
    showarrow=False,
    font=dict(color="#AAAAAA", size=11),
    align="center"
)

# ==========================================
# CONFIGURAÇÃO DO LAYOUT
# ==========================================
fig.update_layout(
    title='<b>Raio-X: Eficiência Ofensiva e Defensiva da Seleção Brasileira</b><br><sup>Volume de gols e saldo histórico por edição</sup>',
    template='plotly_dark',
    barmode='group',
    xaxis=dict(
        title='Ano da Copa', 
        tickmode='array', 
        tickvals=desempenho_por_copa['Year'], 
        tickangle=-45,
        type='linear' 
    ),
    yaxis=dict(title='Quantidade de Gols', gridcolor='#333333', zeroline=True, zerolinecolor='#666666', zerolinewidth=2),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=40, r=40, t=100, b=40),
    hovermode="x unified",
    plot_bgcolor='#111111',
    paper_bgcolor='#111111'
)

fig.show()

**Bloco 2: A dependência de um jogador**

***Funções para normalizar o texto e contar gols***

In [29]:
def normalizar_nome(texto):
    """Remove acentos e padroniza o texto em maiúsculas."""
    if pd.isna(texto): return ""
    texto_normalizado = unicodedata.normalize('NFKD', str(texto))
    return "".join([c for c in texto_normalizado if not unicodedata.combining(c)]).upper().strip()

def contar_gols(evento):
    """Minera a string de eventos buscando G (Gol) ou P (Pênalti)."""
    if pd.isna(evento): return 0
    # Aceita espaços opcionais entre a letra e o minuto do gol
    return len(re.findall(r'[GP]\s*\d+', str(evento)))

***Criando e adequando o Dataframe Unificado para análise***

In [32]:
df_analise = df_brasil_final.copy()

df_analise['Team Name Clean'] = df_analise['Team Name'].astype(str).str.strip().str.upper()
df_analise['Player Name Clean'] = df_analise['Player Name'].apply(normalizar_nome)

# Tratamento específico de sinônimos/variações para atletas chave
df_analise['Player Name Clean'] = df_analise['Player Name Clean'].replace({
    'NEYMAR JR': 'NEYMAR',
    'NEYMAR JR.': 'NEYMAR'
})

***Filtrando pela seleção brasileira e corrigindo duplicatas***

In [33]:
df_jogadores_brasil = df_analise[df_analise['Team Name Clean'].str.contains('^BRA$|^BRAZIL$', na=False, regex=True)].copy()

df_jogadores_brasil = df_jogadores_brasil.drop_duplicates(subset=['MatchID', 'Player Name Clean']).copy()

***Extraindo os gols***

In [34]:
df_jogadores_brasil['Gols na Partida'] = df_jogadores_brasil['Event'].apply(contar_gols)

df_gols_br = df_jogadores_brasil[df_jogadores_brasil['Gols na Partida'] > 0]

gols_por_jogador = df_gols_br.groupby(['Year', 'Player Name Clean'])['Gols na Partida'].sum().reset_index()

***Identificação de artilheiros e tratamento dos empates***

In [35]:
gols_por_jogador['Max Gols Ano'] = gols_por_jogador.groupby('Year')['Gols na Partida'].transform('max')

# Filtra os jogadores que atingiram essa marca máxima
artilheiros_copa = gols_por_jogador[gols_por_jogador['Gols na Partida'] == gols_por_jogador['Max Gols Ano']].copy()

# Consolida a tabela. Em caso de empate, une os nomes com uma barra ' / '
artilheiros_consolidados = artilheiros_copa.groupby('Year').agg({
    'Player Name Clean': lambda x: ' / '.join(sorted(x.unique())),
    'Gols na Partida': 'first'
}).reset_index()

artilheiros_consolidados.rename(columns={
    'Gols na Partida': 'Gols do Artilheiro', 
    'Player Name Clean': 'Artilheiro(s)'
}, inplace=True)

***Calculando a métrica e cruzando com os dados bloco 1***

In [42]:
df_indicadores = pd.merge(
    artilheiros_consolidados, 
    desempenho_por_copa[['Year', 'Gols Feitos']], 
    on='Year', 
    how='inner'
)
df_indicadores.rename(columns={'Gols Feitos': 'Gols do Time'}, inplace=True)

# Forçando o tipo inteiro (Remove o 19.000)
df_indicadores['Dependência (%)'] = (df_indicadores['Gols do Artilheiro'] / df_indicadores['Gols do Time']) * 100
df_indicadores['Year'] = df_indicadores['Year'].astype(int)
df_indicadores['Gols do Time'] = df_indicadores['Gols do Time'].astype(int)
df_indicadores['Gols do Artilheiro'] = df_indicadores['Gols do Artilheiro'].astype(int)

***Tratando os nomes com acento para a visualização***

In [43]:
mapeamento_nomes_dinamico = df_analise.groupby('Player Name Clean')['Player Name'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]
).to_dict()

# 2. Função universal que formata qualquer nome de qualquer seleção
def formatar_nome_escalavel(nome_limpo):
    # Separa caso haja empates (ex: "PELE / VAVA")
    nomes = nome_limpo.split(' / ')
    
    nomes_formatados = []
    for n in nomes:
        # Busca a grafia original no nosso dicionário dinâmico
        nome_original = mapeamento_nomes_dinamico.get(n, n)
        # Aplica a Primeira Letra Maiúscula e guarda
        nomes_formatados.append(nome_original.title())
        
    return ' / '.join(nomes_formatados)

# Aplicamos a função na coluna
df_indicadores['Artilheiro(s)'] = df_indicadores['Artilheiro(s)'].apply(formatar_nome_escalavel)

***Criando a visualização do Bloco 2***

In [41]:
tabela_estilizada = (
    df_indicadores.style
    .format({
        'Dependência (%)': '{:.1f}%', 
        'Year': '{}'                  
    })
    .background_gradient(subset=['Dependência (%)'], cmap='OrRd') 
    .set_properties(**{
        'text-align': 'center',
        'border': '1px solid #dddddd',
        'padding': '8px'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#002776'), ('color', 'white'), ('text-align', 'center'), ('font-size', '14px')]
    }])
    .hide(axis='index') 
)

display(tabela_estilizada)

Year,Artilheiro(s),Gols do Artilheiro,Gols do Time,Dependência (%)
1930,PREGUINHO,3,5.000000,60.0%
1934,LEONIDAS,1,1.000000,100.0%
1938,LEONIDAS,7,14.000000,50.0%
1950,ADEMIR,8,22.000000,36.4%
1954,DIDI / JULINHO / PINGA,2,8.000000,25.0%
1958,PEL� (EDSON ARANTES DO NASCIMENTO),6,16.000000,37.5%
1962,GARRINCHA / VAVA,4,14.000000,28.6%
1966,GARRINCHA / PEL� (EDSON ARANTES DO NASCIMENTO) / RILDO / TOSTAO,1,4.000000,25.0%
1970,JAIRZINHO,7,19.000000,36.8%
1974,RIVELINO,3,6.000000,50.0%
